In [5]:
import os
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# 1. The Dataset Blueprint
class TrafficSignDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.annotations = pd.read_csv(csv_file)
        self.transform = transform
        
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        img_path = self.annotations.iloc[idx]['Path']
        full_path = os.path.abspath(img_path)
        
        image = Image.open(full_path).convert('RGB')
        y_label = torch.tensor(int(self.annotations.iloc[idx]['ClassId']))
        
        if self.transform:
            image = self.transform(image)
            
        return (image, y_label)

# 2. The Model Blueprint
class TrafficSignNet(nn.Module):
    def __init__(self, num_classes=43):
        super(TrafficSignNet, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = torch.flatten(x, 1)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [6]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("Building Custom TrafficSignNet and loading weights...")
model = TrafficSignNet(num_classes=43).to(device)

model.load_state_dict(torch.load('traffic_sign_model.pth', map_location=device))
model.eval() # Turns off dropout

print("Loading Test data...")
transform = transforms.Compose([
    transforms.Resize((32, 32)), 
    transforms.ToTensor()
])

test_dataset = TrafficSignDataset(csv_file='Test.csv', transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=64, shuffle=False)

correct = 0
total = 0

print("Grading all test images... (This will take a few seconds)")

with torch.no_grad(): # No learning allowed
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


final_accuracy = 100 * correct / total
print("-" * 40)
print(f"Official Custom Model Test Accuracy: {final_accuracy:.2f}%")
print(f"Got {correct} out of {total} images perfectly correct!")
print("-" * 40)

Using device: mps
Building Custom TrafficSignNet and loading weights...
Loading Test data...
Grading all test images... (This will take a few seconds)
----------------------------------------
Official Custom Model Test Accuracy: 94.06%
Got 11880 out of 12630 images perfectly correct!
----------------------------------------
